# ⚡ FLUX — Retail Supply Chain Intelligence
> *Predict the rush. Never run out.*

In [ ]:
# ─────────────────────────────────────────────
# CELL 0 — IMPORTS & SETUP
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from scipy.signal import savgol_filter
from scipy import stats
import os

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'monospace',
    'figure.dpi':       130,
})

ACCENT   = '#58a6ff'   # blue
SUCCESS  = '#3fb950'   # green
WARN     = '#d29922'   # yellow
DANGER   = '#f85149'   # red
PURPLE   = '#bc8cff'
TEAL     = '#39d353'

print('✅ Imports ready')

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — LOAD DATA
# Change DATA_PATH to your CSV location
# ─────────────────────────────────────────────
DATA_PATH = 'retail_store_inventory.csv'  # <- update if needed

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Derived columns
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Week']       = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfWeek']  = df['Date'].dt.dayofweek
df['MonthName']  = df['Date'].dt.strftime('%b')
df['YearMonth']  = df['Date'].dt.to_period('M')

print(f'📦 Loaded {len(df):,} rows × {len(df.columns)} columns')
print(f'📅 Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'🏪 Stores: {df["Store ID"].nunique()}  |  🛒 Products: {df["Product ID"].nunique()}  |  📂 Categories: {df["Category"].nunique()}')
df.head(3)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — ✅ DATA QUALITY REPORT
# ─────────────────────────────────────────────────────────────────
print('=' * 60)
print('         ⚡ FLUX  DATA QUALITY REPORT')
print('=' * 60)

total_rows  = len(df)
total_cols  = len(df.columns)
null_counts = df.isnull().sum()
null_pct    = (null_counts / total_rows * 100).round(2)
dupe_rows   = df.duplicated().sum()
neg_sales   = (df['Units Sold'] < 0).sum()
neg_inv     = (df['Inventory Level'] < 0).sum()

# Completeness score
completeness = round(100 - (null_counts.sum() / (total_rows * total_cols) * 100), 2)

print(f'\n📊 Shape          : {total_rows:,} rows × {total_cols} columns')
print(f'📅 Date Range     : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'✅ Completeness   : {completeness}%')
print(f'♻️  Duplicates     : {dupe_rows}')
print(f'⚠️  Negative Sales : {neg_sales}')
print(f'⚠️  Negative Inv.  : {neg_inv}')

print('\n📋 Null Values Per Column:')
for col in df.columns:
    status = '✅' if null_counts[col] == 0 else '❌'
    print(f'   {status} {col:<25} nulls={null_counts[col]}  ({null_pct[col]}%)')

# Outlier detection per numeric column
print('\n🔍 Outlier Detection (Z-score > 3):')
numeric_cols = ['Units Sold','Inventory Level','Demand Forecast','Price','Units Ordered']
for col in numeric_cols:
    z = np.abs(stats.zscore(df[col].dropna()))
    outliers = (z > 3).sum()
    print(f'   {col:<25} outliers={outliers}  ({round(outliers/len(df)*100,2)}%)')

print(f'\n🏆 Overall Data Quality Score: {completeness}% — ', end='')
print('EXCELLENT ✅' if completeness > 98 else 'GOOD 🟡' if completeness > 90 else 'NEEDS CLEANING 🔴')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — ✅ INVENTORY ANALYSIS: TURNOVER, DEAD STOCK, OVERSTOCK
# ─────────────────────────────────────────────────────────────────
inv = df.groupby('Product ID').agg(
    avg_daily_sales  = ('Units Sold',      'mean'),
    avg_inventory    = ('Inventory Level', 'mean'),
    total_sold       = ('Units Sold',      'sum'),
    min_inventory    = ('Inventory Level', 'min'),
    max_inventory    = ('Inventory Level', 'max'),
    std_inventory    = ('Inventory Level', 'std'),
    category         = ('Category',        'first'),
).reset_index()

days_in_data = (df['Date'].max() - df['Date'].min()).days
inv['turnover_ratio']   = inv['total_sold'] / (inv['avg_inventory'] * days_in_data / 365)
inv['days_on_hand']     = inv['avg_inventory'] / inv['avg_daily_sales'].replace(0, np.nan)
inv['dead_stock_flag']  = inv['days_on_hand'] > 60
inv['overstock_flag']   = inv['days_on_hand'] > 30
inv['stockout_risk']    = inv['days_on_hand'] < 7

# Risk label
def risk_label(row):
    if row['stockout_risk']:    return '🔴 CRITICAL'
    if row['days_on_hand'] < 14: return '🟡 WARNING'
    if row['dead_stock_flag']:  return '💀 DEAD STOCK'
    if row['overstock_flag']:   return '📦 OVERSTOCK'
    return '🟢 SAFE'

inv['risk'] = inv.apply(risk_label, axis=1)

print('\n📊 INVENTORY HEALTH SUMMARY')
print(f'  Dead Stock Products  : {inv["dead_stock_flag"].sum()}')
print(f'  Overstock Products   : {inv["overstock_flag"].sum()}')
print(f'  Stockout Risk        : {inv["stockout_risk"].sum()}')
print(f'  Avg Turnover Ratio   : {inv["turnover_ratio"].mean():.2f}x/year')

# ── PLOT ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('⚡ FLUX — Inventory Analysis Dashboard', fontsize=14, color=ACCENT, fontweight='bold', y=1.02)

# Turnover ratio bar chart
ax = axes[0]
colors = [SUCCESS if v > 2 else WARN if v > 1 else DANGER for v in inv['turnover_ratio']]
ax.barh(inv['Product ID'], inv['turnover_ratio'], color=colors, edgecolor='none')
ax.axvline(2.0, color=ACCENT, linestyle='--', linewidth=1, label='Healthy (2x)')
ax.set_title('Inventory Turnover Ratio', color=ACCENT)
ax.set_xlabel('Turns/Year')
ax.legend()
ax.grid(axis='x')

# Days on hand
ax2 = axes[1]
doh_colors = [DANGER if v < 7 else WARN if v < 14 else SUCCESS if v < 30 else '#8b949e' for v in inv['days_on_hand']]
ax2.barh(inv['Product ID'], inv['days_on_hand'], color=doh_colors, edgecolor='none')
ax2.axvline(7,  color=DANGER, linestyle=':', linewidth=1, alpha=0.7, label='Critical (<7d)')
ax2.axvline(30, color=WARN,   linestyle=':', linewidth=1, alpha=0.7, label='Overstock (>30d)')
ax2.set_title('Days On Hand per Product', color=ACCENT)
ax2.set_xlabel('Days')
ax2.legend(fontsize=8)
ax2.grid(axis='x')

# Risk distribution pie
ax3 = axes[2]
risk_counts = inv['risk'].value_counts()
pie_colors  = [SUCCESS, WARN, DANGER, PURPLE, '#8b949e']
wedges, texts, autotexts = ax3.pie(
    risk_counts.values,
    labels=risk_counts.index,
    autopct='%1.0f%%',
    colors=pie_colors[:len(risk_counts)],
    startangle=90,
    textprops={'fontsize': 8},
)
for t in autotexts: t.set_color('white')
ax3.set_title('Product Risk Distribution', color=ACCENT)

plt.tight_layout()
plt.savefig('flux_inventory_analysis.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()
print('\n📋 Inventory Table:')
display(inv[['Product ID','category','avg_daily_sales','avg_inventory','days_on_hand','turnover_ratio','risk']].round(1))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — ✅ DEMAND PATTERNS: SEASONALITY, TRENDS, VOLATILITY
# ─────────────────────────────────────────────────────────────────
daily   = df.groupby('Date')['Units Sold'].sum().reset_index()
monthly = df.groupby(['Year','Month','MonthName'])['Units Sold'].sum().reset_index()
monthly['period'] = monthly['Year'].astype(str) + '-' + monthly['Month'].apply(lambda x: str(x).zfill(2))
monthly = monthly.sort_values('period')

cat_monthly  = df.groupby(['Category','YearMonth'])['Units Sold'].sum().reset_index()
season_sales = df.groupby('Seasonality')['Units Sold'].mean().reset_index()
weekday_sales= df.groupby('DayOfWeek')['Units Sold'].mean().reset_index()
weekday_sales['DayName'] = weekday_sales['DayOfWeek'].map({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'})

fig = plt.figure(figsize=(20, 14))
fig.suptitle('⚡ FLUX — Demand Pattern Analysis', fontsize=15, color=ACCENT, fontweight='bold')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Total daily sales with trend
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(daily['Date'], daily['Units Sold'], alpha=0.2, color=ACCENT)
ax1.plot(daily['Date'], daily['Units Sold'], color=ACCENT, linewidth=0.8, alpha=0.7)
# Rolling mean trend
rolling = daily['Units Sold'].rolling(30, min_periods=1).mean()
ax1.plot(daily['Date'], rolling, color=SUCCESS, linewidth=2, label='30-day trend')
ax1.set_title('Total Daily Sales — Full History', color=ACCENT)
ax1.set_xlabel('Date')
ax1.set_ylabel('Units Sold')
ax1.legend()
ax1.grid(True)

# 2. Monthly sales by category
ax2 = fig.add_subplot(gs[1, :2])
cat_colors = [ACCENT, SUCCESS, WARN, DANGER, PURPLE]
for i, cat in enumerate(df['Category'].unique()):
    sub = cat_monthly[cat_monthly['Category']==cat].sort_values('YearMonth')
    ax2.plot(sub['YearMonth'].astype(str), sub['Units Sold'],
             label=cat, color=cat_colors[i], linewidth=1.5, marker='o', markersize=3)
ax2.set_title('Monthly Sales by Category', color=ACCENT)
ax2.set_xlabel('Month')
ax2.set_ylabel('Units Sold')
ax2.legend(fontsize=8)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True)

# 3. Seasonality
ax3 = fig.add_subplot(gs[1, 2])
season_order = ['Spring','Summer','Autumn','Winter']
s_data = season_sales.set_index('Seasonality').reindex(season_order).reset_index()
bars = ax3.bar(s_data['Seasonality'], s_data['Units Sold'],
               color=[ACCENT, SUCCESS, WARN, PURPLE], edgecolor='none', width=0.6)
ax3.set_title('Avg Sales by Season', color=ACCENT)
ax3.set_ylabel('Avg Units Sold/Day')
for bar, val in zip(bars, s_data['Units Sold']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}', ha='center', va='bottom', fontsize=9, color='white')
ax3.grid(axis='y')

# 4. Day of week pattern
ax4 = fig.add_subplot(gs[2, 0])
w_colors = [WARN if d >= 5 else ACCENT for d in weekday_sales['DayOfWeek']]
ax4.bar(weekday_sales['DayName'], weekday_sales['Units Sold'], color=w_colors, edgecolor='none')
ax4.set_title('Sales by Day of Week', color=ACCENT)
ax4.set_ylabel('Avg Units Sold')
ax4.grid(axis='y')

# 5. Weather impact
ax5 = fig.add_subplot(gs[2, 1])
weather_avg = df.groupby('Weather Condition')['Units Sold'].mean().sort_values()
w_clrs = [ACCENT, TEAL, WARN, PURPLE]
ax5.barh(weather_avg.index, weather_avg.values, color=w_clrs, edgecolor='none')
ax5.set_title('Avg Sales by Weather', color=ACCENT)
ax5.set_xlabel('Avg Units Sold')
ax5.grid(axis='x')

# 6. Holiday / Promotion impact
ax6 = fig.add_subplot(gs[2, 2])
holiday_avg = df.groupby('Holiday/Promotion')['Units Sold'].mean()
ax6.bar(['Normal Day','Holiday/Promo'], holiday_avg.values,
        color=[ACCENT, WARN], edgecolor='none', width=0.5)
lift = (holiday_avg[1] / holiday_avg[0] - 1) * 100
ax6.set_title(f'Holiday Demand Lift: +{lift:.1f}%', color=SUCCESS)
ax6.set_ylabel('Avg Units Sold')
ax6.grid(axis='y')

plt.savefig('flux_demand_patterns.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()
print(f'\n📈 Holiday demand lift: +{lift:.1f}% vs normal days')
print(f'🌤️  Best weather for sales: {weather_avg.idxmax()} ({weather_avg.max():.0f} units/day avg)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — ✅ DEMAND SPIKE DETECTION & VISUALIZATION
# Like Akkio / enterprise inventory software
# ─────────────────────────────────────────────────────────────────
def detect_spikes(series: pd.Series, window: int = 7, threshold: float = 1.5) -> pd.Series:
    rolling_mean = series.rolling(window, min_periods=1).mean()
    spike_score  = series / rolling_mean.replace(0, np.nan)
    return spike_score

# Aggregate all stores per product per day
pivot = df.groupby(['Date','Product ID'])['Units Sold'].sum().unstack('Product ID').fillna(0)

# Pick 4 representative products
sample_products = pivot.columns[:4].tolist()

fig, axes = plt.subplots(2, 2, figsize=(20, 10))
fig.suptitle('⚡ FLUX — Demand Spike Detection', fontsize=15, color=ACCENT, fontweight='bold')
axes = axes.flatten()

all_spikes_summary = []

for i, pid in enumerate(sample_products):
    ax = axes[i]
    series = pivot[pid].copy()
    smoothed   = pd.Series(savgol_filter(series.values, 11, 3), index=series.index)
    spike_score = detect_spikes(series)
    spike_mask  = spike_score > 1.5
    dip_mask    = spike_score < 0.5
    rolling_mu  = series.rolling(7, min_periods=1).mean()
    rolling_std = series.rolling(7, min_periods=1).std().fillna(0)

    # Confidence band
    upper = rolling_mu + 1.5 * rolling_std
    lower = (rolling_mu - 1.5 * rolling_std).clip(lower=0)

    ax.fill_between(series.index, lower, upper, alpha=0.12, color=ACCENT, label='Normal band')
    ax.plot(series.index, series,  color='#484f58', linewidth=0.7, alpha=0.8)
    ax.plot(series.index, smoothed, color=ACCENT, linewidth=1.5, label='Smoothed')

    # Spike markers
    if spike_mask.any():
        ax.scatter(series.index[spike_mask], series[spike_mask],
                   color=DANGER, s=40, zorder=5, label=f'Spikes ({spike_mask.sum()})', marker='^')
    if dip_mask.any():
        ax.scatter(series.index[dip_mask], series[dip_mask],
                   color=WARN, s=30, zorder=5, label=f'Dips ({dip_mask.sum()})', marker='v')

    cat = df[df['Product ID']==pid]['Category'].iloc[0]
    ax.set_title(f'{pid} ({cat})  |  Max Spike: {spike_score.max():.1f}x baseline', color=ACCENT)
    ax.set_ylabel('Units Sold')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True)

    all_spikes_summary.append({
        'Product': pid, 'Category': cat,
        'Total Spikes': spike_mask.sum(),
        'Max Spike Score': round(spike_score.max(), 2),
        'Avg Dip Score':   round(spike_score.min(), 2),
        'Volatility (CV)': round(series.std() / series.mean() * 100, 1),
    })

plt.tight_layout()
plt.savefig('flux_spike_detection.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()

spikes_df = pd.DataFrame(all_spikes_summary)
print('\n📊 Spike Summary:')
display(spikes_df)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — ✅ HEATMAP: SPIKE CALENDAR (like Akkio/software UIs)
# ─────────────────────────────────────────────────────────────────
# Monthly spike intensity heatmap across all products
df_daily_cat = df.groupby(['Date','Category'])['Units Sold'].sum().reset_index()
df_daily_cat['MonthYear'] = df_daily_cat['Date'].dt.to_period('M').astype(str)

monthly_cat = df_daily_cat.groupby(['Category','MonthYear'])['Units Sold'].sum().unstack('MonthYear').fillna(0)

# Normalize each row to show relative intensity
monthly_norm = monthly_cat.div(monthly_cat.mean(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(22, 4))
fig.patch.set_facecolor('#0d1117')

cmap = LinearSegmentedColormap.from_list('flux', ['#0d1117', '#1f6feb', '#58a6ff', '#f85149'])
im = ax.imshow(monthly_norm.values, aspect='auto', cmap=cmap, vmin=0.7, vmax=1.4)

ax.set_yticks(range(len(monthly_norm.index)))
ax.set_yticklabels(monthly_norm.index, fontsize=10)
ax.set_xticks(range(len(monthly_norm.columns)))
ax.set_xticklabels(monthly_norm.columns, rotation=45, ha='right', fontsize=7)
ax.set_title('⚡ FLUX — Monthly Demand Intensity Heatmap (Relative to Mean)', fontsize=13, color=ACCENT, pad=12)

cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.01, pad=0.01)
cbar.set_label('Demand Multiplier vs Monthly Avg', color='#8b949e', fontsize=8)
cbar.ax.yaxis.set_tick_params(color='#8b949e', labelcolor='#8b949e')

# Annotate cells with multiplier
for r in range(monthly_norm.shape[0]):
    for c in range(monthly_norm.shape[1]):
        val = monthly_norm.iloc[r, c]
        color = 'white' if val > 1.1 else '#8b949e'
        ax.text(c, r, f'{val:.1f}x', ha='center', va='center', fontsize=5.5, color=color)

plt.tight_layout()
plt.savefig('flux_spike_heatmap.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()
print('🔥 Cells > 1.2x = demand spikes  |  Cells < 0.85x = demand dips')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 — ✅ DEMAND FORECAST WITH CONFIDENCE INTERVALS
# Polynomial regression + rolling baseline (no Prophet needed)
# ─────────────────────────────────────────────────────────────────
FORECAST_PRODUCT = 'P0001'   # <- change this to any product ID
FORECAST_DAYS    = 30

# Aggregate across all stores for selected product
prod_daily = (
    df[df['Product ID'] == FORECAST_PRODUCT]
    .groupby('Date')['Units Sold']
    .sum()
    .reset_index()
    .sort_values('Date')
)

# Smooth the signal
prod_daily['smoothed'] = pd.Series(
    savgol_filter(prod_daily['Units Sold'].values, 15, 3)
).clip(lower=0).values

# Train/test split — last 30 days as holdout
train = prod_daily.iloc[:-FORECAST_DAYS].copy()
test  = prod_daily.iloc[-FORECAST_DAYS:].copy()

train['t'] = np.arange(len(train))
X_train = train[['t']].values
y_train = train['smoothed'].values

# Polynomial regression (degree 4)
model = Pipeline([('poly', PolynomialFeatures(degree=4)), ('lr', LinearRegression())])
model.fit(X_train, y_train)

# Predict on test + 30 days ahead
t_test    = np.arange(len(train), len(train) + FORECAST_DAYS)
t_future  = np.arange(len(train) + FORECAST_DAYS, len(train) + FORECAST_DAYS * 2)

y_pred_test   = model.predict(t_test.reshape(-1, 1)).clip(min=0)
y_pred_future = model.predict(t_future.reshape(-1, 1)).clip(min=0)

# Residual std for confidence band
y_train_pred = model.predict(X_train)
residuals    = y_train - y_train_pred
resid_std    = residuals.std()

# Accuracy on test set
mape = mean_absolute_percentage_error(test['Units Sold'].values, y_pred_test) * 100
mae  = mean_absolute_error(test['Units Sold'].values, y_pred_test)

# Future dates
last_date    = prod_daily['Date'].iloc[-1]
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS)

# ── PLOT ──
fig, axes = plt.subplots(2, 1, figsize=(20, 11), gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle(f'⚡ FLUX — Demand Forecast: {FORECAST_PRODUCT}  |  MAPE: {mape:.1f}%  |  MAE: {mae:.0f} units',
             fontsize=13, color=ACCENT, fontweight='bold')

ax = axes[0]
# Historical
ax.fill_between(prod_daily['Date'], prod_daily['Units Sold'], alpha=0.08, color=ACCENT)
ax.plot(prod_daily['Date'], prod_daily['Units Sold'], color='#484f58', linewidth=0.7, alpha=0.8, label='Actual')
ax.plot(train['Date'], y_train_pred, color=ACCENT, linewidth=1.5, linestyle='--', alpha=0.7, label='Fit')

# Test prediction
ax.plot(test['Date'], y_pred_test, color=SUCCESS, linewidth=2, label='Validation Forecast')
ax.fill_between(test['Date'],
                y_pred_test - 1.96 * resid_std,
                y_pred_test + 1.96 * resid_std,
                alpha=0.15, color=SUCCESS, label='95% Confidence')

# Future forecast
ax.plot(future_dates, y_pred_future, color=WARN, linewidth=2.5, label=f'Future ({FORECAST_DAYS}d)', linestyle='-')
ax.fill_between(future_dates,
                (y_pred_future - 1.96 * resid_std).clip(min=0),
                y_pred_future + 1.96 * resid_std,
                alpha=0.2, color=WARN)

# Vertical dividers
ax.axvline(test['Date'].iloc[0],  color='#30363d', linestyle='--', linewidth=1)
ax.axvline(future_dates[0],       color=WARN,      linestyle='--', linewidth=1, alpha=0.6)
ax.text(future_dates[0], ax.get_ylim()[1] * 0.97, '  Forecast →', color=WARN, fontsize=9)

ax.set_ylabel('Units Sold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True)

# Residuals
ax2 = axes[1]
resid_test = test['Units Sold'].values - y_pred_test
ax2.bar(test['Date'], resid_test,
        color=[DANGER if r < 0 else SUCCESS for r in resid_test],
        alpha=0.7, edgecolor='none')
ax2.axhline(0, color='#30363d', linewidth=1)
ax2.set_ylabel('Residuals')
ax2.set_xlabel('Date')
ax2.set_title('Forecast Residuals (Actual − Predicted)', color='#8b949e', fontsize=9)
ax2.grid(True)

plt.tight_layout()
plt.savefig('flux_forecast.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()

print(f'\n📊 Forecast Accuracy:')
print(f'   MAPE  : {mape:.2f}%  ({"EXCELLENT ✅" if mape < 10 else "GOOD 🟡" if mape < 20 else "REVIEW 🔴"})')
print(f'   MAE   : {mae:.1f} units')
print(f'\n🔮 30-Day Forecast Summary:')
print(f'   Total predicted demand : {y_pred_future.sum():.0f} units')
print(f'   Daily avg              : {y_pred_future.mean():.0f} units')
print(f'   Peak day               : {future_dates[y_pred_future.argmax()].date()} ({y_pred_future.max():.0f} units)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 — ✅ FORECAST ACCURACY SCORE — ALL PRODUCTS
# ─────────────────────────────────────────────────────────────────
accuracy_results = []

for pid in df['Product ID'].unique():
    prod = (
        df[df['Product ID'] == pid]
        .groupby('Date')['Units Sold']
        .sum()
        .reset_index()
        .sort_values('Date')
    )
    if len(prod) < 60:
        continue
    prod['smoothed'] = pd.Series(savgol_filter(prod['Units Sold'].values, 15, 3)).clip(lower=0).values
    split = int(len(prod) * 0.8)
    train_p = prod.iloc[:split].copy()
    test_p  = prod.iloc[split:].copy()
    train_p['t'] = np.arange(len(train_p))
    t_test  = np.arange(len(train_p), len(prod))
    m = Pipeline([('poly', PolynomialFeatures(degree=3)), ('lr', LinearRegression())])
    m.fit(train_p[['t']].values, train_p['smoothed'].values)
    y_pred = m.predict(t_test.reshape(-1, 1)).clip(min=0)
    try:
        mape_p = mean_absolute_percentage_error(test_p['Units Sold'].values, y_pred) * 100
    except:
        mape_p = 999
    accuracy_results.append({
        'Product ID': pid,
        'Category': df[df['Product ID']==pid]['Category'].iloc[0],
        'MAPE %': round(mape_p, 2),
        'Confidence': '✅ High' if mape_p < 10 else '🟡 Medium' if mape_p < 20 else '🔴 Low',
    })

acc_df = pd.DataFrame(accuracy_results).sort_values('MAPE %')

fig, ax = plt.subplots(figsize=(14, 6))
colors = [SUCCESS if m < 10 else WARN if m < 20 else DANGER for m in acc_df['MAPE %']]
bars = ax.barh(acc_df['Product ID'], acc_df['MAPE %'], color=colors, edgecolor='none')
ax.axvline(10, color=SUCCESS, linestyle='--', linewidth=1, label='High Confidence (<10%)')
ax.axvline(20, color=WARN,    linestyle='--', linewidth=1, label='Medium (<20%)')
for bar, val in zip(bars, acc_df['MAPE %']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)
ax.set_title('⚡ FLUX — Forecast Accuracy (MAPE) per Product', color=ACCENT, fontsize=13)
ax.set_xlabel('MAPE % (lower = better)')
ax.legend(fontsize=8)
ax.grid(axis='x')
plt.tight_layout()
plt.savefig('flux_accuracy.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()

print(f'\n📊 Overall Model Accuracy:')
print(f'   Mean MAPE : {acc_df["MAPE %"].mean():.2f}%')
print(f'   Best      : {acc_df.iloc[0]["Product ID"]} at {acc_df.iloc[0]["MAPE %"]:.2f}%')
print(f'   Worst     : {acc_df.iloc[-1]["Product ID"]} at {acc_df.iloc[-1]["MAPE %"]:.2f}%')
display(acc_df)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 9 — ✅ HISTORICAL COMPARISON: Month vs Month, Quarter vs Quarter
# ─────────────────────────────────────────────────────────────────
df['Quarter'] = df['Date'].dt.quarter
df['YearQ']   = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)

monthly_total = df.groupby(['Year','Month'])['Units Sold'].sum().reset_index()
qtr_total     = df.groupby(['YearQ'])['Units Sold'].sum().reset_index().sort_values('YearQ')

# MoM and QoQ growth
monthly_total = monthly_total.sort_values(['Year','Month'])
monthly_total['MoM_Growth'] = monthly_total['Units Sold'].pct_change() * 100
qtr_total['QoQ_Growth']     = qtr_total['Units Sold'].pct_change() * 100

fig, axes = plt.subplots(2, 2, figsize=(20, 10))
fig.suptitle('⚡ FLUX — Historical Comparison Report', fontsize=14, color=ACCENT, fontweight='bold')

# Monthly sales grouped by year
ax1 = axes[0, 0]
months = range(1, 13)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
year_colors = {2022: ACCENT, 2023: SUCCESS, 2024: WARN}
for yr in monthly_total['Year'].unique():
    yd = monthly_total[monthly_total['Year'] == yr]
    ax1.plot(yd['Month'], yd['Units Sold'],
             label=str(yr), color=year_colors.get(yr, PURPLE),
             linewidth=2, marker='o', markersize=5)
ax1.set_xticks(list(months))
ax1.set_xticklabels(month_names)
ax1.set_title('Monthly Sales: Year-over-Year', color=ACCENT)
ax1.set_ylabel('Units Sold')
ax1.legend()
ax1.grid(True)

# MoM growth
ax2 = axes[0, 1]
mom = monthly_total.dropna(subset=['MoM_Growth'])
mom_label = mom['Year'].astype(str) + '-' + mom['Month'].apply(lambda x: str(x).zfill(2))
bar_colors = [SUCCESS if g > 0 else DANGER for g in mom['MoM_Growth']]
ax2.bar(range(len(mom)), mom['MoM_Growth'], color=bar_colors, edgecolor='none')
ax2.set_xticks(range(0, len(mom), 3))
ax2.set_xticklabels(mom_label.iloc[::3], rotation=45, ha='right', fontsize=7)
ax2.axhline(0, color='#30363d', linewidth=1)
ax2.set_title('Month-over-Month Growth %', color=ACCENT)
ax2.set_ylabel('Growth %')
ax2.grid(axis='y')

# Quarterly total
ax3 = axes[1, 0]
q_colors = [ACCENT if '2022' in q else SUCCESS if '2023' in q else WARN for q in qtr_total['YearQ']]
ax3.bar(qtr_total['YearQ'], qtr_total['Units Sold'], color=q_colors, edgecolor='none')
for i, (_, row) in enumerate(qtr_total.iterrows()):
    ax3.text(i, row['Units Sold'] + 5000, f'{row["Units Sold"]/1e6:.2f}M',
             ha='center', fontsize=8, color='white')
ax3.set_title('Quarterly Sales Total', color=ACCENT)
ax3.set_ylabel('Units Sold')
ax3.tick_params(axis='x', rotation=30)
ax3.grid(axis='y')

# QoQ growth
ax4 = axes[1, 1]
qoq = qtr_total.dropna(subset=['QoQ_Growth'])
bar_colors2 = [SUCCESS if g > 0 else DANGER for g in qoq['QoQ_Growth']]
ax4.bar(qoq['YearQ'], qoq['QoQ_Growth'], color=bar_colors2, edgecolor='none')
ax4.axhline(0, color='#30363d', linewidth=1)
ax4.set_title('Quarter-over-Quarter Growth %', color=ACCENT)
ax4.set_ylabel('Growth %')
ax4.tick_params(axis='x', rotation=30)
ax4.grid(axis='y')

plt.tight_layout()
plt.savefig('flux_historical_comparison.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()

best_month = monthly_total.loc[monthly_total['Units Sold'].idxmax()]
print(f'\n📈 Best month ever: {int(best_month["Year"])}-{int(best_month["Month"]):02d}  ({int(best_month["Units Sold"]):,} units)')
print(f'📊 Best quarter  : {qtr_total.loc[qtr_total["Units Sold"].idxmax(), "YearQ"]}  ({int(qtr_total["Units Sold"].max()):,} units)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 10 — ✅ RISK CLASSIFICATION DASHBOARD (Final Output)
# ─────────────────────────────────────────────────────────────────
# Use last 30 days as 'current' state
last_30 = df[df['Date'] >= df['Date'].max() - pd.Timedelta(days=30)]

summary = last_30.groupby('Product ID').agg(
    avg_daily_sales  = ('Units Sold',      'mean'),
    current_stock    = ('Inventory Level', 'last'),
    category         = ('Category',        'first'),
    holiday_days     = ('Holiday/Promotion','sum'),
).reset_index()

summary['days_to_stockout'] = (summary['current_stock'] / summary['avg_daily_sales']).round(1)
summary['reorder_qty']      = np.ceil((summary['avg_daily_sales'] * 14) - summary['current_stock']).clip(lower=0).astype(int)

def classify(days):
    if days <= 3:  return '🔴 CRITICAL',  DANGER
    if days <= 7:  return '🟡 WARNING',   WARN
    if days <= 14: return '🔵 MONITOR',   ACCENT
    return '🟢 SAFE', SUCCESS

summary['risk_label'], summary['risk_color'] = zip(*summary['days_to_stockout'].apply(classify))
summary = summary.sort_values('days_to_stockout')

fig, ax = plt.subplots(figsize=(16, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.set_xlim(0, 1)
ax.set_ylim(0, len(summary) + 1)
ax.axis('off')
ax.set_title('⚡ FLUX — Risk Classification Dashboard (Last 30 Days)',
             fontsize=14, color=ACCENT, fontweight='bold', pad=15)

headers = ['Product', 'Category', 'Stock', 'Daily Sales', 'Days Left', 'Reorder Qty', 'Status']
col_x   = [0.02, 0.17, 0.32, 0.45, 0.57, 0.70, 0.84]

for j, h in enumerate(headers):
    ax.text(col_x[j], len(summary) + 0.5, h, fontsize=9,
            color=ACCENT, fontweight='bold', va='center')

for i, (_, row) in enumerate(summary.iterrows()):
    y = len(summary) - i - 0.5
    bg_color = '#161b22' if i % 2 == 0 else '#0d1117'
    rect = mpatches.FancyBboxPatch((0, y - 0.4), 1, 0.8,
                                    boxstyle='round,pad=0.01',
                                    facecolor=bg_color, edgecolor='none')
    ax.add_patch(rect)
    values = [
        row['Product ID'],
        row['category'],
        f"{int(row['current_stock']):,}",
        f"{row['avg_daily_sales']:.0f}/day",
        f"{row['days_to_stockout']:.1f}d",
        f"{int(row['reorder_qty'])} units" if row['reorder_qty'] > 0 else '—',
        row['risk_label'],
    ]
    for j, val in enumerate(values):
        color = row['risk_color'] if j == 6 else '#c9d1d9'
        if j == 6: color = row['risk_color']
        ax.text(col_x[j], y, str(val), fontsize=8.5, va='center', color=color)

plt.tight_layout()
plt.savefig('flux_risk_dashboard.png', bbox_inches='tight', dpi=150, facecolor='#0d1117')
plt.show()

critical = summary[summary['risk_label'].str.contains('CRITICAL')]
warning  = summary[summary['risk_label'].str.contains('WARNING')]
print(f'\n🔴 CRITICAL ({len(critical)} products): {list(critical["Product ID"])}')
print(f'🟡 WARNING  ({len(warning)} products): {list(warning["Product ID"])}')
print(f'🟢 SAFE     ({len(summary) - len(critical) - len(warning)} products)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 11 — ✅ EXPORT TO CSV + EXCEL
# ─────────────────────────────────────────────────────────────────
import os

# Prepare export DataFrames
export_risk = summary[['Product ID','category','current_stock','avg_daily_sales',
                        'days_to_stockout','reorder_qty','risk_label']].copy()
export_acc  = acc_df.copy()
export_inv  = inv[['Product ID','category','avg_daily_sales','avg_inventory',
                    'days_on_hand','turnover_ratio','risk']].copy()

# CSV exports
export_risk.to_csv('flux_risk_report.csv', index=False)
export_acc.to_csv('flux_accuracy_report.csv', index=False)
export_inv.to_csv('flux_inventory_report.csv', index=False)
print('✅ CSV reports saved:')
print('   📄 flux_risk_report.csv')
print('   📄 flux_accuracy_report.csv')
print('   📄 flux_inventory_report.csv')

# Excel export (multi-sheet)
try:
    with pd.ExcelWriter('FLUX_Report.xlsx', engine='openpyxl') as writer:
        export_risk.to_excel(writer, sheet_name='Risk Dashboard', index=False)
        export_acc.to_excel(writer, sheet_name='Forecast Accuracy', index=False)
        export_inv.to_excel(writer, sheet_name='Inventory Analysis', index=False)
        monthly_total.to_excel(writer, sheet_name='Monthly Comparison', index=False)
    print('\n✅ Excel report saved: FLUX_Report.xlsx (4 sheets)')
except ImportError:
    print('⚠️  openpyxl not installed — pip install openpyxl for Excel export')

print('\n📁 All output files:')
for f in ['flux_inventory_analysis.png','flux_demand_patterns.png','flux_spike_detection.png',
          'flux_spike_heatmap.png','flux_forecast.png','flux_accuracy.png',
          'flux_historical_comparison.png','flux_risk_dashboard.png',
          'flux_risk_report.csv','flux_accuracy_report.csv','flux_inventory_report.csv','FLUX_Report.xlsx']:
    exists = '✅' if os.path.exists(f) else '❌'
    print(f'   {exists} {f}')

---
## 🌐 CELL 12 — Free Real-Time API Integration Guide

### Plug these into FLUX for live demand signals:

| API | What it gives FLUX | Free Tier | Student Access |
|-----|-------------------|-----------|----------------|
| **OpenWeatherMap** | 7-day weather forecast by pincode → demand adjustments | 1,000 calls/day free | ✅ Same free tier |
| **Ticketmaster Discovery API** | Local events near store → spike multipliers | Unlimited read (free) | ✅ Free |
| **Google Calendar API** | National holidays for India | Free (with Google account) | ✅ Free |
| **Open-Meteo** | Weather forecast — no API key needed at all | Unlimited free | ✅ No signup |
| **Predicthq** | Events + holidays + school terms | 100 events/month free | ✅ Student plan available |
| **NewsAPI** | Local news that signals demand (elections, disasters) | 100 req/day free | ✅ Student plan (free) |
| **Alpha Vantage** | Commodity prices (oil, wheat) → cost forecasting | 25 req/day free | ✅ Free |
| **Data.gov.in** | India-specific agricultural and FMCG price data | Fully open | ✅ Free |

### Quickstart: Open-Meteo (zero API key)
```python
import requests
# Chennai coords: 13.0827, 80.2707
url = 'https://api.open-meteo.com/v1/forecast?latitude=13.08&longitude=80.27&daily=temperature_2m_max,precipitation_sum&timezone=Asia/Kolkata&forecast_days=7'
data = requests.get(url).json()
temps = data['daily']['temperature_2m_max']   # list of 7 max temps
rain  = data['daily']['precipitation_sum']    # list of 7 rain totals
```

### Quickstart: Ticketmaster (free key)
```python
# Register at developer.ticketmaster.com → free API key in 2 mins
API_KEY = 'your_key_here'
url = f'https://app.ticketmaster.com/discovery/v2/events.json?city=Chennai&radius=10&unit=km&apikey={API_KEY}'
events = requests.get(url).json()['_embedded']['events']
```

### Quickstart: PredictHQ (student free tier)
```python
# Sign up at predicthq.com → verify student email → 100 events/month free
headers = {'Authorization': 'Bearer YOUR_TOKEN'}
url = 'https://api.predicthq.com/v1/events/?country=IN&q=festival&limit=10'
events = requests.get(url, headers=headers).json()
```

---
## 🔮 Prophet Upgrade Path
When you have internet access, upgrade forecasting to Prophet for sub-5% MAPE:
```bash
pip install prophet
```
```python
from prophet import Prophet
# Your existing prod_daily DataFrame works directly:
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, interval_width=0.80)
m.fit(prod_daily.rename(columns={'Date':'ds','Units Sold':'y'}))
future = m.make_future_dataframe(periods=30)
forecast = m.predict(future)
m.plot(forecast)
```
The rest of FLUX (spike detection, risk classification, export) works unchanged.

---
*⚡ FLUX — Predict the rush. Never run out.*